# Inspect Rejected Records

## Spark setup cell

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("InspectRejectedRecords")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.session.timeZone", "UTC")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 16:55:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Inspect individual rejected records

In [2]:
REJECTED_RECORDS_PATH = (
    "s3a://delta-data/metadata/rejected_records"
)

rejected_records = (
    spark.read
    .format("delta")
    .load(REJECTED_RECORDS_PATH)
)

rejected_records.printSchema()

(
    rejected_records
    .orderBy(F.desc("rejected_at"))
    .show(100, truncate=False)
)

26/09/27 16:55:12 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- pipeline_run_id: string (nullable = true)
 |-- execution_id: string (nullable = true)
 |-- stage: string (nullable = true)
 |-- dataset: string (nullable = true)
 |-- schema_version: string (nullable = true)
 |-- rejected_at: timestamp (nullable = true)
 |-- rejection_reasons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- raw_record: string (nullable = true)



26/09/27 16:55:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------------------------+------------------------------------+------+----------+--------------+-------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|pipeline_run_id                     |execution_id                        |stage |dataset   |schema_version|rejected_at         

## Summarize failures by reason

In [3]:
(
    rejected_records
    .select(
        "dataset",
        "schema_version",
        F.explode("rejection_reasons").alias(
            "rejection_reason"
        ),
    )
    .groupBy(
        "dataset",
        "schema_version",
        "rejection_reason",
    )
    .count()
    .orderBy(
        F.desc("count"),
        "dataset",
        "rejection_reason",
    )
    .show(truncate=False)
)

+----------+--------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+
|dataset   |schema_version|rejection_reason                                                                                                                                                                                                                                                                                                          |count |
+----------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Stop Spark

In [4]:
spark.stop()